# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q Faker statsmodels
    print('Running in Colab')

# Logistic Regression In-Depth

Predicting binary outcomes (Voting Behavior) using Logistic Regression. We will use `statsmodels` for statistical inference (Odds Ratios) and `scikit-learn` for predictive evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker

np.random.seed(42)

# Generate Synthetic Voting Data
n = 500
age = np.random.randint(18, 90, n)
income = np.random.normal(50000, 20000, n)
educ = np.random.randint(10, 22, n) # Years of education

# Define underlying relationship (Log-odds)
# Older, richer, and more educated people are more likely to vote
logits = -12 + 0.08 * age + 0.00004 * income + 0.4 * educ
probs = 1 / (1 + np.exp(-logits))

# Binary outcome based on probability
voted = [1 if np.random.rand() < p else 0 for p in probs]

df = pd.DataFrame({'Age': age, 'Income': income, 'Education_Years': educ, 'Voted': voted})
print(df.head())
print("\nVoting Rate:", df['Voted'].mean())

## 1. Statistical Inference (Statsmodels)
Using Statsmodels to interpret coefficients and calculate Odds Ratios.

In [ ]:
import statsmodels.api as sm

# Define predictors and target
X = df[['Age', 'Income', 'Education_Years']]
y = df['Voted']

# Add constant for the intercept
X_const = sm.add_constant(X)

# Fit Logistic Regression
model = sm.Logit(y, X_const).fit()
print(model.summary())

### Odds Ratios
Coefficients in logistic regression are log-odds. Exponentiating them gives the Odds Ratio (OR), which is easier to interpret.

In [ ]:
odds_ratios = np.exp(model.params)
conf = np.exp(model.conf_int())
conf['Odds Ratio'] = odds_ratios
conf.columns = ['CI Lower', 'CI Upper', 'Odds Ratio']
print(conf)

## 2. Predictive Performance (Scikit-Learn)
 evaluating the model using accuracy, confusion matrix, and ROC curve.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Fit Model
clf = LogisticRegression()
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

# Metrics
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### ROC Curve and AUC
The Receiver Operating Characteristic (ROC) curve plots the True Positive Rate against the False Positive Rate.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()